# YOLO + OCR Plate Detection - Colab Notebook

- install the dependencies
- load the YOLO model
- detect the plates
- read text using EasyOCR
- process a single image and a full folder
- save the results in an `outputs` folder

In [ ]:
# Cell 1 - Install dependencies
# Run this cell when starting the notebook.

!pip install -q ultralytics easyocr opencv-python-headless numpy pandas

print('Dependencies installed.')

In [ ]:
# Cell 2 - Check the repository/project
# If the repo is not present, clone it from GitHub.

import os
from pathlib import Path

REPO_URL = "https://github.com/senisama/AI-UAV-Address-Verification.git"
WORKDIR = Path("/content")
PROJECT_DIR = WORKDIR / "AI-UAV-Address-Verification"

if not PROJECT_DIR.exists():
    %cd /content
    !git clone {REPO_URL}

%cd {PROJECT_DIR}

print(f"Project available at: {PROJECT_DIR}")
print("Directory tree:")
!ls -la

In [ ]:
# Cell 3 - Define the working paths

from pathlib import Path

PROJECT_ROOT = Path('/content/AI-UAV-Address-Verification')
MODEL_PATH = PROJECT_ROOT / 'models' / 'best.pt'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('MODEL_PATH =', MODEL_PATH)
print('RAW_DIR =', RAW_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert RAW_DIR.exists(), f"Raw folder not found: {RAW_DIR}"

In [ ]:
# Cell 4 - Import the pipeline

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from plate_ocr.detect_and_ocr import PlateOCRPipeline

print('Pipeline imported successfully.')

In [ ]:
# Cell 5 - Initialize the pipeline

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

pipeline = PlateOCRPipeline(
    model_path=MODEL_PATH,
    languages=('en',),
    device=DEVICE,
)

print('YOLO + EasyOCR are ready.')

In [ ]:
# Cell 6 - Test on a single image

image_path = RAW_DIR / 'images' / 'test1.jpg'

if not image_path.exists():
    image_candidates = list((RAW_DIR / 'images').glob('*.*'))
    if image_candidates:
        image_path = image_candidates[0]
        print('Default image used:', image_path)
    else:
        raise FileNotFoundError(f"No image found in {RAW_DIR / 'images'}")

print('Processing:', image_path)
detections, output_path = pipeline.detect_and_read(image_path, conf_threshold=0.25)

print(f"Number of detections: {len(detections)}")
print('Annotated image saved to:', output_path)

for i, det in enumerate(detections, start=1):
    print(f"Plate {i}: bbox={det['bbox']} confidence={det['confidence']:.3f} texts={det['texts']}")

from IPython.display import Image, display

display(Image(filename=str(output_path), width=800))

In [ ]:
# Cell 7 - Process a full folder

from pathlib import Path

results = pipeline.detect_and_read_directory(RAW_DIR / 'images', conf_threshold=0.25)

print(f"Images processed: {len(results)}")
for item in results:
    print(item['image'], '->', len(item['detections']), 'detection(s)')
    for det in item['detections']:
        print('  ', det['bbox'], det['texts'])

In [ ]:
# Cell 8 - Export results to a table

import pandas as pd

rows = []
for item in results:
    for det in item['detections']:
        rows.append({
            'image': item['image'],
            'bbox': det['bbox'],
            'confidence': det['confidence'],
            'texts': ' | '.join(det.get('texts', [])),
            'output': str(item['output'])
        })

if rows:
    df = pd.DataFrame(rows)
    print(df.head())
    summary_path = PROJECT_ROOT / 'outputs' / 'results_summary.csv'
    df.to_csv(summary_path, index=False)
    print('CSV saved to:', summary_path)
else:
    print('No detections found.')

In [ ]:
# Cell 9 - Process a video
# Replace the path below with your own video file if needed.

video_path = PROJECT_ROOT / 'data' / 'raw' / 'example_video.mp4'
output_video_path = PROJECT_ROOT / 'outputs' / 'annotated_video.mp4'

if video_path.exists():
    print('Processing video:', video_path)
    frame_count = pipeline.process_video(
        video_source=str(video_path),
        output_path=str(output_video_path),
        conf_threshold=0.25,
        show=False,
        max_frames=100,
        fps=25,
    )
    print(f'Processed {frame_count} frames and saved output to: {output_video_path}')
else:
    print('No video file found at:', video_path)
    print('Add a video such as data/raw/example_video.mp4 or change the path above.')